# Stretch energy calculation and its diravatives

In [76]:
import sympy as sp
from sympy import symbols, integrate, expand

In [77]:
xs = symbols('x_{1:10}', real=True)
x = sp.Matrix(xs)
x

Matrix([
[x_{1}],
[x_{2}],
[x_{3}],
[x_{4}],
[x_{5}],
[x_{6}],
[x_{7}],
[x_{8}],
[x_{9}]])

In [78]:
Dm_tmp = symbols('D_m11 D_m21 D_m12 D_m22 ', real=True)
Dm_tmp = sp.Matrix(Dm_tmp)
Dm = sp.Matrix.zeros(2,2)
Dm[0:2,0] = Dm_tmp[0:2]
Dm[0:2,1] = Dm_tmp[2:4]
Dm

Matrix([
[D_m11, D_m12],
[D_m21, D_m22]])

In [79]:
x1 = sp.Matrix(x[0:3, 0])
x2 = sp.Matrix(x[3:6, 0])
x3 = sp.Matrix(x[6:9, 0])

In [80]:
Ds = sp.Matrix.zeros(3,2)
Ds[0:3,0] = x2 - x1
Ds[0:3,1] = x3 - x1
Ds

Matrix([
[-x_{1} + x_{4}, -x_{1} + x_{7}],
[-x_{2} + x_{5}, -x_{2} + x_{8}],
[-x_{3} + x_{6}, -x_{3} + x_{9}]])

In [81]:
F = Ds * Dm
F

Matrix([
[D_m11*(-x_{1} + x_{4}) + D_m21*(-x_{1} + x_{7}), D_m12*(-x_{1} + x_{4}) + D_m22*(-x_{1} + x_{7})],
[D_m11*(-x_{2} + x_{5}) + D_m21*(-x_{2} + x_{8}), D_m12*(-x_{2} + x_{5}) + D_m22*(-x_{2} + x_{8})],
[D_m11*(-x_{3} + x_{6}) + D_m21*(-x_{3} + x_{9}), D_m12*(-x_{3} + x_{6}) + D_m22*(-x_{3} + x_{9})]])

In [82]:
F_vec = F.vec() # Do not use F.reshape(6, 1) since it will change the order of elements
F_vec

Matrix([
[D_m11*(-x_{1} + x_{4}) + D_m21*(-x_{1} + x_{7})],
[D_m11*(-x_{2} + x_{5}) + D_m21*(-x_{2} + x_{8})],
[D_m11*(-x_{3} + x_{6}) + D_m21*(-x_{3} + x_{9})],
[D_m12*(-x_{1} + x_{4}) + D_m22*(-x_{1} + x_{7})],
[D_m12*(-x_{2} + x_{5}) + D_m22*(-x_{2} + x_{8})],
[D_m12*(-x_{3} + x_{6}) + D_m22*(-x_{3} + x_{9})]])

In [83]:
dFdx = (F_vec).jacobian(x)
dFdx

Matrix([
[-D_m11 - D_m21,              0,              0, D_m11,     0,     0, D_m21,     0,     0],
[             0, -D_m11 - D_m21,              0,     0, D_m11,     0,     0, D_m21,     0],
[             0,              0, -D_m11 - D_m21,     0,     0, D_m11,     0,     0, D_m21],
[-D_m12 - D_m22,              0,              0, D_m12,     0,     0, D_m22,     0,     0],
[             0, -D_m12 - D_m22,              0,     0, D_m12,     0,     0, D_m22,     0],
[             0,              0, -D_m12 - D_m22,     0,     0, D_m12,     0,     0, D_m22]])

In [84]:
import importlib

import dfdx_codegen_utils as dcu
importlib.reload(dcu)
print_dfdx_codegen = dcu.print_dfdx_codegen

# ===== Config =====
PY_SCALAR_TYPE = 'float'
EMIT_ZERO_ENTRIES = False
# RowMajor / ColumnMajor / Both
MATRIX_LAYOUT = 'Both'

print_dfdx_codegen(
    dfdx=dFdx,
    dm_symbol_names=['D_m11', 'D_m21', 'D_m12', 'D_m22'],
    scalar_type=PY_SCALAR_TYPE,
    emit_zero_entries=EMIT_ZERO_ENTRIES,
    layout=MATRIX_LAYOUT,
    mapping={
        'D_m11': 'd0',
        'D_m21': 'd1',
        'D_m12': 'd2',
        'D_m22': 'd3',
    },
    rowmajor_name='get_dFdx_RowMajor',
    columnmajor_name='get_dFdx_ColumnMajor',
)

=== Python Function ===
def build_dFdx(D_m11: float, D_m21: float, D_m12: float, D_m22: float):
    return ImmutableDenseMatrix([[-D_m11 - D_m21, 0, 0, D_m11, 0, 0, D_m21, 0, 0], [0, -D_m11 - D_m21, 0, 0, D_m11, 0, 0, D_m21, 0], [0, 0, -D_m11 - D_m21, 0, 0, D_m11, 0, 0, D_m21], [-D_m12 - D_m22, 0, 0, D_m12, 0, 0, D_m22, 0, 0], [0, -D_m12 - D_m22, 0, 0, D_m12, 0, 0, D_m22, 0], [0, 0, -D_m12 - D_m22, 0, 0, D_m12, 0, 0, D_m22]])

=== C++ RowMajor Interface (style like get_dFdx) ===
inline LargeMatrix<6, 9> get_dFdx_RowMajor(const luisa::float2x2& InverseDm)
{
    const float d0 = InverseDm[0][0];
    const float d1 = InverseDm[0][1];
    const float d2 = InverseDm[1][0];
    const float d3 = InverseDm[1][1];

    LargeMatrix<6, 9> result_rm = LargeMatrix<6, 9>::zero();
    const float val_1 = -d0 - d1;
    const float val_2 = -d2 - d3;
    set_matrix_scalar(result_rm, 0, 0, val_1);
    set_matrix_scalar(result_rm, 0, 3, d0);
    set_matrix_scalar(result_rm, 0, 6, d1);
    set_matrix_scala

##  For Finite-Element Formulation of Baraff-Witkin Cloth

In [85]:
# Vector2 anisotropic_a = Vector2(1, 0);
# Vector2 anisotropic_b = Vector2(0, 1);
a = sp.Matrix([1, 0])
b = sp.Matrix([0, 1])
I6 = a.transpose() * F.transpose() * F * b
shear_energy = I6 * I6
shear_energy

Matrix([[((D_m11*(-x_{1} + x_{4}) + D_m21*(-x_{1} + x_{7}))*(D_m12*(-x_{1} + x_{4}) + D_m22*(-x_{1} + x_{7})) + (D_m11*(-x_{2} + x_{5}) + D_m21*(-x_{2} + x_{8}))*(D_m12*(-x_{2} + x_{5}) + D_m22*(-x_{2} + x_{8})) + (D_m11*(-x_{3} + x_{6}) + D_m21*(-x_{3} + x_{9}))*(D_m12*(-x_{3} + x_{6}) + D_m22*(-x_{3} + x_{9})))**2]])

In [86]:
shear_energy.jacobian(x).transpose()

Matrix([
[(2*(-D_m11 - D_m21)*(D_m12*(-x_{1} + x_{4}) + D_m22*(-x_{1} + x_{7})) + 2*(-D_m12 - D_m22)*(D_m11*(-x_{1} + x_{4}) + D_m21*(-x_{1} + x_{7})))*((D_m11*(-x_{1} + x_{4}) + D_m21*(-x_{1} + x_{7}))*(D_m12*(-x_{1} + x_{4}) + D_m22*(-x_{1} + x_{7})) + (D_m11*(-x_{2} + x_{5}) + D_m21*(-x_{2} + x_{8}))*(D_m12*(-x_{2} + x_{5}) + D_m22*(-x_{2} + x_{8})) + (D_m11*(-x_{3} + x_{6}) + D_m21*(-x_{3} + x_{9}))*(D_m12*(-x_{3} + x_{6}) + D_m22*(-x_{3} + x_{9})))],
[(2*(-D_m11 - D_m21)*(D_m12*(-x_{2} + x_{5}) + D_m22*(-x_{2} + x_{8})) + 2*(-D_m12 - D_m22)*(D_m11*(-x_{2} + x_{5}) + D_m21*(-x_{2} + x_{8})))*((D_m11*(-x_{1} + x_{4}) + D_m21*(-x_{1} + x_{7}))*(D_m12*(-x_{1} + x_{4}) + D_m22*(-x_{1} + x_{7})) + (D_m11*(-x_{2} + x_{5}) + D_m21*(-x_{2} + x_{8}))*(D_m12*(-x_{2} + x_{5}) + D_m22*(-x_{2} + x_{8})) + (D_m11*(-x_{3} + x_{6}) + D_m21*(-x_{3} + x_{9}))*(D_m12*(-x_{3} + x_{6}) + D_m22*(-x_{3} + x_{9})))],
[(2*(-D_m11 - D_m21)*(D_m12*(-x_{3} + x_{6}) + D_m22*(-x_{3} + x_{9})) + 2*(-D_m12 - D_m2

In [87]:
I5u = (F * a).norm()
I5v = (F * b).norm()

ucoeff = sp.Piecewise((0, I5u <= 1), (1, True))
vcoeff = sp.Piecewise((0, I5v <= 1), (1, True))
strainRate = symbols('strainRate', real=True)
stretch_energy = (I5u - 1)**2 + ucoeff * strainRate * (I5u - 1)**3 + (I5v - 1)**2 + vcoeff * strainRate * (I5v - 1)**3
stretch_energy


strainRate*(sqrt((D_m11*(x_{1} - x_{4}) + D_m21*(x_{1} - x_{7}))**2 + (D_m11*(x_{2} - x_{5}) + D_m21*(x_{2} - x_{8}))**2 + (D_m11*(x_{3} - x_{6}) + D_m21*(x_{3} - x_{9}))**2) - 1)**3*Piecewise((0, sqrt((D_m11*(x_{1} - x_{4}) + D_m21*(x_{1} - x_{7}))**2 + (D_m11*(x_{2} - x_{5}) + D_m21*(x_{2} - x_{8}))**2 + (D_m11*(x_{3} - x_{6}) + D_m21*(x_{3} - x_{9}))**2) <= 1), (1, True)) + strainRate*(sqrt((D_m12*(x_{1} - x_{4}) + D_m22*(x_{1} - x_{7}))**2 + (D_m12*(x_{2} - x_{5}) + D_m22*(x_{2} - x_{8}))**2 + (D_m12*(x_{3} - x_{6}) + D_m22*(x_{3} - x_{9}))**2) - 1)**3*Piecewise((0, sqrt((D_m12*(x_{1} - x_{4}) + D_m22*(x_{1} - x_{7}))**2 + (D_m12*(x_{2} - x_{5}) + D_m22*(x_{2} - x_{8}))**2 + (D_m12*(x_{3} - x_{6}) + D_m22*(x_{3} - x_{9}))**2) <= 1), (1, True)) + (sqrt((D_m11*(x_{1} - x_{4}) + D_m21*(x_{1} - x_{7}))**2 + (D_m11*(x_{2} - x_{5}) + D_m21*(x_{2} - x_{8}))**2 + (D_m11*(x_{3} - x_{6}) + D_m21*(x_{3} - x_{9}))**2) - 1)**2 + (sqrt((D_m12*(x_{1} - x_{4}) + D_m22*(x_{1} - x_{7}))**2 + (D_m12*

## Tmp

In [88]:
E, nu = symbols('E nu', real=True) 


C_mat = (E/(1-nu**2)) * sp.Matrix([
    [1, nu, 0],
    [nu, 1, 0], 
    [0, 0, (1-nu)/2]
])


# E = 0.5*(F^T F - I)
I = sp.eye(2)
E_green = 0.5 * (F.T * F - I)

epsilon = sp.Matrix([E_green[0,0], E_green[1,1], 2*E_green[0,1]])
sigma = C_mat * epsilon

print("sigma:")
sp.pprint(sigma)


dsigmadx = sigma.jacobian(x)
print("\nd sigma d x :", dsigmadx.shape)

sigma:
⎡    ⎛                                                   2                     ↪
⎢E⋅ν⋅⎝0.5⋅(Dₘ₁₂⋅(-x_{1} + x_{4}) + Dₘ₂₂⋅(-x_{1} + x_{7}))  + 0.5⋅(Dₘ₁₂⋅(-x_{2} ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢    ⎛                                                   2                     ↪
⎢E⋅ν⋅⎝0.5⋅(Dₘ₁₁⋅(-x_{1} + x_{4}) + Dₘ₂₁⋅(-x_{1} + x_{7}))  + 0.5⋅(Dₘ₁₁⋅(-x_{2} ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                   ⎛